# Create the shared `funder_reported_work_funders` table (union owner)

**Union owner for the `from_funder_reported` leg** of
`notebooks/end2end/CreateWorksEnriched.ipynb` (oxjob #739, Casey-approved 2026-08-10;
originally assigned to Kyle). This notebook is the single scheduled place that rebuilds
`openalex.awards.funder_reported_work_funders` — do not copy this cell into a source
job (e.g. `CreateHakaiWorkFunders`, `CreateDataCiteWorkFunders`); add a new source by
adding one `UNION` branch here.

**Why this moved out of `CreateHakaiWorkFunders`:** that notebook's Step 3 used to own
this table, but `Hakai_Work_Funders` (the job) is PAUSED by design — Hakai's own
scrape+resolve steps are on-demand OUTPUT-LIST work, not a nightly job (see
`jobs/hakai_work_funders.yaml`). Scheduling that whole notebook nightly to keep this
union fresh would silently re-run the Hakai scrape-staging steps every night, which is
exactly the "wasted work" the pause was written to avoid. This notebook contains only
the union rebuild, so it can run nightly on its own schedule
(`jobs/funder_reported_work_funders.yaml`) without touching any upstream source job.

**Sources unioned (provenance-tagged, one row per `(work_id, funder_id, provenance)`):**
`hakai_work_funders`, `europepmc_work_funders`, `datacite_work_funders`,
`kaken_work_funders`, `anr_work_funders`. Each source stays owned and scheduled (or
run on-demand) by its own notebook; this notebook only reads their output tables.

**Additive by design:** this is a `MERGE ... WHEN NOT MATCHED THEN INSERT`, not a
`CREATE OR REPLACE`. A destructive rebuild would drop any row whose source table has
since had a pair retracted (measured 2026-08-10: 1,741 such EPMC pairs) and silently
remove a public `work.funders` label. If source retractions should ever propagate as
removals, that is a separate owner-approved policy decision, not a side effect of
keeping this table fresh. See `wait-investigation-20260810/ENRICHMENT-READY/SUMMARY.md`
for the full investigation. Gates re-run fresh 2026-08-14 against the materialized DEV
candidate (Gate A v2, replacement-semantics guard model: 0 uncovered, 0 uncovered
keep-list-gated; Gate B: 0 old pairs lost, 487,913 net-new; NULL-key and dup-triple
prechecks all 0).

#### Build — additive merge of every funder-reported source

#### Pre-MERGE gates (task FAILS on any violation)

These run BEFORE the merge and abort the task via `assert_true`, so a bad source night never reaches the served table (review finding 2026-08-24: plain SELECT sanity cells cannot fail a job). Each gate names the condition it enforces.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW funder_reported_source_rows AS
SELECT work_id, funder_id, provenance FROM openalex.awards.hakai_work_funders
UNION SELECT work_id, funder_id, 'europepmc_work_funders' FROM openalex.awards.europepmc_work_funders
UNION SELECT work_id, funder_id, 'datacite_work_funders'  FROM openalex.awards.datacite_work_funders
UNION SELECT work_id, funder_id, 'kaken_work_funders'     FROM openalex.awards.kaken_work_funders
UNION SELECT work_id, funder_id, 'anr_work_funders'       FROM openalex.awards.anr_work_funders;


In [ ]:
%sql
-- GATE 1: no NULL key in any source row. A NULL key can never MERGE-match itself
-- (NULL = NULL is not TRUE) and would be re-inserted every night.
SELECT assert_true(
  COUNT_IF(work_id IS NULL OR funder_id IS NULL OR provenance IS NULL) = 0,
  'funder_reported: NULL key in source rows') AS gate_null_keys
FROM funder_reported_source_rows;

In [ ]:
%sql
-- GATE 2: every source funder_id resolves in mid.funder AND is not a merge loser.
-- CreateWorksEnriched inner-joins mid.funder; a merged-away funder would publish a
-- retired id (MergeFunders does not rewrite this table). Expect 0 / 0.
SELECT assert_true(
  COUNT_IF(f.funder_id IS NULL) = 0,
  'funder_reported: source funder_id missing from mid.funder') AS gate_orphan_funder,
  assert_true(
  COUNT_IF(f.merge_into_id IS NOT NULL) = 0,
  'funder_reported: source funder_id is a merge loser (run MergeFunders remap first)') AS gate_merge_loser
FROM (SELECT DISTINCT funder_id FROM funder_reported_source_rows) s
LEFT JOIN openalex.mid.funder f ON f.funder_id = s.funder_id;

In [ ]:
%sql
-- GATE 3: per-run insertion ceiling. Net-new triples vs the served table must stay
-- under 5,000,000. The first attended run inserts ~3.24M (candidate 17,127,098 minus the
-- frozen union 13,884,308, measured 2026-08-24); steady-state nightly deltas are ~1k
-- (DataCite v80->v81 churn = +925/-3). A full source-table replacement would be >13M. A larger jump means a source table was
-- replaced with something unexpected; stop and look before it becomes permanent.
SELECT assert_true(
  COUNT(*) < 5000000,
  'funder_reported: net-new triples exceed the per-run ceiling') AS gate_insert_ceiling
FROM funder_reported_source_rows s
LEFT ANTI JOIN openalex.awards.funder_reported_work_funders t
  ON t.work_id = s.work_id AND t.funder_id = s.funder_id AND t.provenance = s.provenance;

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS openalex.awards.funder_reported_work_funders (
  work_id BIGINT,
  funder_id BIGINT,
  provenance STRING
)
USING DELTA;

MERGE INTO openalex.awards.funder_reported_work_funders AS target
USING funder_reported_source_rows AS source
-- NULL-key guard: a NULL in any key column can never MERGE-match itself on a later
-- run (NULL = NULL is not TRUE), so such a row would be re-inserted every night.
-- Source tables shouldn't produce NULL keys, but this makes the merge idempotent by
-- construction rather than by upstream promise.
ON  target.work_id = source.work_id
AND target.funder_id = source.funder_id
AND target.provenance = source.provenance
WHEN NOT MATCHED
  AND source.work_id IS NOT NULL
  AND source.funder_id IS NOT NULL
  AND source.provenance IS NOT NULL
THEN INSERT (work_id, funder_id, provenance)
VALUES (source.work_id, source.funder_id, source.provenance);

#### Sanity checks

In [ ]:
%sql
-- Row + distinct-pair counts by provenance. distinct_pairs may be lower than rows if a
-- (work_id, funder_id) pair is asserted by more than one source (kept as separate rows
-- by design -- provenance stays auditable per source).
SELECT
  provenance,
  COUNT(*)                                          AS rows,
  COUNT(DISTINCT CONCAT(work_id, ':', funder_id))   AS distinct_pairs
FROM openalex.awards.funder_reported_work_funders
GROUP BY provenance
ORDER BY provenance;

In [ ]:
%sql
-- Every funder_id must resolve in the dim table (CreateWorksEnriched's from_funder_reported
-- leg inner-joins on this). Expect 0 rows.
SELECT DISTINCT fr.funder_id
FROM openalex.awards.funder_reported_work_funders fr
LEFT ANTI JOIN openalex.mid.funder f ON f.funder_id = fr.funder_id;

In [ ]:
%sql
-- HARD CHECK 1: zero NULL-key rows in the served table. A non-empty result here means
-- a NULL key slipped in historically and the table needs a one-time cleanup before the
-- merge's idempotency guarantee holds. Expect 0.
SELECT assert_true(COUNT(*) = 0, 'funder_reported: NULL-key rows in served table') AS null_key_rows
FROM openalex.awards.funder_reported_work_funders
WHERE work_id IS NULL OR funder_id IS NULL OR provenance IS NULL;


In [ ]:
%sql
-- HARD CHECK 2: zero duplicate (work_id, funder_id, provenance) triples. The MERGE key
-- makes duplicates impossible going forward; a non-zero count means pre-existing dups.
-- Expect 0. (Re-running the MERGE must also insert 0 rows -- check num_inserted_rows in
-- the merge cell's output on any manual re-run.)
SELECT assert_true(COUNT(*) = 0, 'funder_reported: duplicate triples in served table') AS dup_triples
FROM (
  SELECT work_id, funder_id, provenance
  FROM openalex.awards.funder_reported_work_funders
  GROUP BY work_id, funder_id, provenance
  HAVING COUNT(*) > 1
);


In [ ]:
%sql
-- INFORMATIONAL (not a gate): cross-provenance overlap -- how many (work_id, funder_id)
-- pairs are asserted by more than one source. These are kept as separate rows by design;
-- CreateWorksEnriched's dedup (GROUP BY work_id, funder_id) collapses them before the
-- public work.funders surface, so overlap here never produces duplicate API funders.
SELECT
  COUNT(*)                                        AS total_rows,
  COUNT(DISTINCT CONCAT(work_id, ':', funder_id)) AS distinct_pairs,
  COUNT(*) - COUNT(DISTINCT CONCAT(work_id, ':', funder_id)) AS cross_provenance_overlap_rows
FROM openalex.awards.funder_reported_work_funders;


#### Schedule

Owned by `jobs/funder_reported_work_funders.yaml`, scheduled nightly at 02:30 UTC --
after `Crossref_Work_Funders` and `DataCite_Work_Funders` (both 02:00 UTC, the only two
inputs here with a genuine live nightly feed) and before `RefreshWorkAwards` (03:00 UTC)
and `Walden_End_2_End` (05:00 UTC, which runs `CreateWorksEnriched` and needs this table
fresh). Hakai / EuropePMC / KAKEN / ANR stay on-demand OUTPUT-LIST sources (see their own
notebooks; KAKEN has no job yaml at all -- its table is rebuilt on demand by
`CreateKAKENWorkAwards`); this job re-unions them on every run regardless, so a manual refresh of
any one of them is picked up the same night without any change here.